In [ ]:
import pickle

with open("transcripts_and_texts.pickle", "rb") as f:
    transcripts = pickle.load(f)

In [ ]:
import torch
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")

In [ ]:
data = ["I love you", "I hate you"]
sentiment_pipeline(data)

In [ ]:
data = ["I don't like you, I absolutely adore you", "I hate you", "this is like the best story ever", "this is the best story ever"]
test = sentiment_pipeline(data)
test

In [ ]:
import polars as pl

# now we apply this
session_sents = transcripts.filter(
    pl.col("speech").str.len_chars() > 10
).with_columns(
    pl.when(pl.col("event") == "Reading Story")
    .then(pl.col("speech").str.split(by="."))
    .otherwise(pl.col("speech").cast(pl.List(pl.Utf8))).alias("speech")
).with_columns(
    pl.col("speech").map_elements(lambda t: sentiment_pipeline(t.to_list()), return_dtype=pl.List(pl.Struct([pl.Field("label", pl.Utf8), pl.Field("score", pl.Float32)]))).alias("sentiment")
)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import polars as pl

In [ ]:
analyser = SentimentIntensityAnalyzer()


In [ ]:
test1 = analyser.polarity_scores(data[0])
test1

In [ ]:
session_sents_vader = transcripts.with_columns(
    pl.col("speech").map_elements(lambda t: analyser.polarity_scores(t), 
                                  return_dtype=pl.Struct([pl.Field("neg", pl.Float64), pl.Field("neu", pl.Float64), pl.Field("pos", pl.Float64), pl.Field("compound", pl.Float64)])).alias("sentiment")
)

In [ ]:
session_sents_vader_parts = transcripts.group_by("session", "title", "index", maintain_order=True).agg(
    speech = pl.col("speech").str.join(" ")
).with_columns(
    pl.col("speech").map_elements(lambda t: analyser.polarity_scores(t), 
                                  return_dtype=pl.Struct([pl.Field("neg", pl.Float64), pl.Field("neu", pl.Float64), pl.Field("pos", pl.Float64), pl.Field("compound", pl.Float64)])).alias("sentiment")
)

In [ ]:
session_sents_vader_parts = session_sents_vader_parts.unnest("sentiment")

In [ ]:
session_sents_vader